# DDF depth: cumulative 5-sigma depth versus time of the Deep Drilling Fields, in baseline survey v5.3

This notebook follows the **coadded depth** of each **Deep Drilling Field (DDF)** of a simulated LSST survey as the visits accumulate, band by band. It is the depth counterpart of `04_DDF_NVisitsVsTime`, which counts the visits. In the baseline v5.3 the *Ocean* DDF strategy alternates shallow and deep seasons for each DDF, so the depth curves grow in steps rather than linearly.

- last reorganisation : 2026-10-10
- companion notebooks: `01_SurveyModes` (DDF positions), `02_SurveyAreas` (DDF visit map), `04_DDF_NVisitsVsTime` (number of visits and cadence of the DDFs)
- Ocean DDF strategy discussion: https://community.lsst.org/t/ddf-observations-under-the-ocean-observing-strategy/12370

## Overview and outputs

1. **Load the DDF visits** from the OpSim database: a visit is attributed to a DDF when its `scheduler_note` (or `note`, `fieldName`) contains a DD tag and the name of one of the five DDFs (COSMOS, EDFS_a, EDFS_b, XMM-LSS, ELAIS-S1, ECDFS; the two EDFS pointings are different sky positions and are kept separate).
2. **Cumulative depth**: for each DDF and each band, combine the single-visit 5-sigma depths in time order.
3. **Summary tables**: number of visits, median single-visit depth and final coadded depth for each DDF and band.
4. **Figure 1**: cumulative depth versus date, one panel per DDF, one curve per band, with the survey years marked.

**Method.** The visits are assumed to be independent and of equal noise properties, so the inverse-variance flux weights add: the coadded depth after visits 1...k of one band is

`m5_cum = 1.25 * log10( sum_i 10**(0.8 * m5_i) )`

(computed with weights relative to the deepest visit to avoid numerical overflow). It is the same combination as the MAF `Coaddm5Metric`. It assumes that all the visits cover the same sky position (the dithers are ignored); the sky-brightness and seeing variations are already included in each single-visit `fiveSigmaDepth`.

**Input**: the OpSim database `baseline_v5.3.6_10yrs.db` (variable `opsdb`).

**Outputs**: the tables (`.csv`) go to `DATA05_DDF_DepthVsTime/`; the figure (`.png` and `.pdf`) goes to `FIG05_DDF_DepthVsTime/`.

## 0. Setup

In [ ]:
import os
import re
import sqlite3
from datetime import date, datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from rubin_scheduler.utils import SURVEY_START_MJD

In [ ]:
# Output directories: tables go to DATA_DIR, figures go to FIG_DIR
DATA_DIR = "DATA05_DDF_DepthVsTime"
FIG_DIR = "FIG05_DDF_DepthVsTime"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# OpSim database. Replace by an explicit path to analyse another run.
# opsdb = get_baseline()
opsdb = "/Users/dagoret/DATA/OpSim/baseline_v5.3.6_10yrs.db"
opsim_name = os.path.split(opsdb)[-1].replace(".db", "")
print(opsdb)

# Calendar date of the survey start, used for the date axis and the year boundaries (change if needed)
SURVEY_START = date(2026, 6, 29)
N_YEARS = 10
YEAR = 365.25  # days

BANDS = list("ugrizy")

# Standard LSST band colours (the y band is drawn in black)
BAND_COLORS = {
    "u": "#4477AA",
    "g": "#228833",
    "r": "#CC3311",
    "i": "#EE7733",
    "z": "#AA3377",
    "y": "k",
}

# DDFs shown in the figure and in the tables, in this order
DDF_NAMES = ["COSMOS", "EDFS_a", "EDFS_b", "XMM-LSS", "ELAIS-S1", "ECDFS"]

### Utility functions

- `mjd_to_datetime`: Modified Julian Date to Python datetime.
- `survey_year_date(n)`: calendar date of the beginning of survey year n+1.
- `identify_ddf`: name of the DDF written in a `scheduler_note`, or `None`. A DD tag (`DD` or `deep drilling`) is required, so that a field with a similar name that is not a DDF visit is not selected.

The last line checks that `SURVEY_START` is consistent with `SURVEY_START_MJD` of the scheduler.

In [ ]:
def mjd_to_datetime(mjd):
    # Convert a Modified Julian Date to a Python datetime
    epoch = datetime(1858, 11, 17)
    return epoch + timedelta(days=float(mjd))


def survey_year_date(n):
    # Calendar date marking the beginning of survey year n+1
    try:
        return SURVEY_START.replace(year=SURVEY_START.year + n)
    except ValueError:
        # 29 February when SURVEY_START is changed to such a date
        return SURVEY_START.replace(year=SURVEY_START.year + n, day=28)


def identify_ddf(text):
    # Identify a DDF from the scheduler_note, note or fieldName of a visit
    if not isinstance(text, str):
        return None

    s = text.upper().replace("_", "-")

    # Regular expression of each DDF name, allowing the common OpSim spellings
    aliases = {
        "COSMOS": r"COSMOS",
        "ECDFS": r"ECDFS",
        # EDFS has two pointings at different sky positions: they must not be coadded together.
        # EDFS alone is a fallback when the note does not say a or b (it then appears in the warning below).
        "EDFS_a": r"EDFS-?A(?![A-Z])",
        "EDFS_b": r"EDFS-?B(?![A-Z])",
        "EDFS": r"EDFS",
        "ELAIS-S1": r"ELAIS[- ]?S?1|ELAISS1",
        "XMM-LSS": r"XMM[- ]?LSS|XMMLSS",
    }

    # A DD tag is required, to avoid selecting an unrelated field with a similar name
    if not re.search(r"\bDD\b|DEEP.?DRILL", s):
        return None

    for name, pattern in aliases.items():
        if re.search(pattern, s):
            return name

    return None


print(
    f"SURVEY_START_MJD = {SURVEY_START_MJD} -> {mjd_to_datetime(SURVEY_START_MJD).date()}; SURVEY_START set above: {SURVEY_START}"
)

## 1. Load the DDF visits

The database is opened in read-only mode. The band column is `band` or `filter` depending on the OpSim version; it is renamed `filter` here. Only the visits attributed to a DDF, in one of the six bands and with a valid time and depth, are kept, sorted in time.

In [ ]:
def load_ddf_visits(db_path):
    # Read the DDF visits of an OpSim SQLite database (read-only)
    db_path = Path(db_path).expanduser().resolve()

    if not db_path.is_file():
        raise FileNotFoundError(f"OpSim database not found: {db_path}")

    # Open an existing database without creating or modifying it
    uri = db_path.as_uri() + "?mode=ro"

    with sqlite3.connect(uri, uri=True) as con:
        tables = pd.read_sql_query(
            "SELECT name FROM sqlite_master WHERE type = ? ORDER BY name", con, params=("table",)
        )["name"].tolist()
        if "observations" not in tables:
            raise RuntimeError(f"Table observations not found. Available tables: {tables}")

        columns = pd.read_sql_query("PRAGMA table_info(observations)", con)["name"].tolist()

        # Band column: depends on the OpSim version
        band_col = next((c for c in ("band", "filter") if c in columns), None)
        if band_col is None:
            raise RuntimeError(f"Neither band nor filter in observations. Columns: {columns}")

        required = ["observationStartMJD", "fiveSigmaDepth"]
        missing = [c for c in required if c not in columns]
        if missing:
            raise RuntimeError(f"Missing required columns: {missing}")

        # Columns in which the name of the DDF may be written
        note_cols = [c for c in ("scheduler_note", "note", "fieldName") if c in columns]
        if not note_cols:
            raise RuntimeError("No scheduler_note, note or fieldName column.")

        selected = list(dict.fromkeys(required + [band_col] + note_cols))
        df = pd.read_sql_query("SELECT " + ", ".join(selected) + " FROM observations", con)

    # Same column name for the rest of the notebook
    df = df.rename(columns={band_col: "filter"})

    df["ddf"] = None
    for col in note_cols:
        df["ddf"] = df["ddf"].fillna(df[col].apply(identify_ddf))

    df["observationStartMJD"] = pd.to_numeric(df["observationStartMJD"], errors="coerce")
    df["fiveSigmaDepth"] = pd.to_numeric(df["fiveSigmaDepth"], errors="coerce")
    df = df.dropna(subset=["ddf", "observationStartMJD", "fiveSigmaDepth"])
    df = df[df["filter"].isin(BANDS)].copy()
    return df.sort_values("observationStartMJD").reset_index(drop=True)


df = load_ddf_visits(opsdb)
print(f"{len(df)} DDF visits")
print(df["ddf"].value_counts())

# A DDF found in the database but absent from DDF_NAMES is not shown in the figure and the tables
extra = sorted(set(df["ddf"]) - set(DDF_NAMES))
if extra:
    print("WARNING: DDFs found but not in DDF_NAMES:", extra)

## 2. Cumulative depth

For each (DDF, band) the visits are taken in time order and the coadded depth after each visit is computed with the formula given in the overview. The result is stored in `df_cum` (columns `years` since the survey start, `n_cum` number of visits so far, `m5_cum` coadded depth, `date`), and saved to `DATA05_DDF_DepthVsTime/` (`..._ddf_cumulative_depth_per_visit.csv`).

In [ ]:
def add_cumulative_depth(group):
    # Coadded depth after each visit, for the visits of ONE DDF in ONE band:
    # m5_cum = 1.25 log10(sum_i 10**(0.8 m5_i)), with weights relative to the deepest visit
    g = group.sort_values("observationStartMJD").copy()
    m5_max = g["fiveSigmaDepth"].max()
    weights = 10.0 ** (0.8 * (g["fiveSigmaDepth"].to_numpy() - m5_max))
    g["n_cum"] = np.arange(1, len(g) + 1)
    g["m5_cum"] = m5_max + 1.25 * np.log10(np.cumsum(weights))
    g["years"] = (g["observationStartMJD"] - SURVEY_START_MJD) / YEAR
    g["date"] = g["observationStartMJD"].apply(mjd_to_datetime)
    return g


df_cum = pd.concat([add_cumulative_depth(g) for _, g in df.groupby(["ddf", "filter"])])
df_cum = df_cum.sort_values("observationStartMJD").reset_index(drop=True)

cols = ["ddf", "filter", "observationStartMJD", "years", "n_cum", "fiveSigmaDepth", "m5_cum"]
df_cum[cols].to_csv(os.path.join(DATA_DIR, f"{opsim_name}_ddf_cumulative_depth_per_visit.csv"), index=False)
df_cum[cols].head()

## 3. Summary tables

For each DDF and band: the number of visits, the median single-visit depth and the **final coadded depth** (after all the visits of the survey). The first table gives the number of visits, the second the final coadded depth (mag); they are saved to `DATA05_DDF_DepthVsTime/`, together with the long table that also has the median single-visit depth and the depth gain (final coadded depth minus median single-visit depth).

In [ ]:
rows = []
for (ddf, band), g in df_cum.groupby(["ddf", "filter"]):
    median_single = g["fiveSigmaDepth"].median()
    final = g["m5_cum"].iloc[-1]
    rows.append(
        {
            "DDF": ddf,
            "band": band,
            "N visits": len(g),
            "median single-visit m5": median_single,
            "final coadded m5": final,
            "gain (mag)": final - median_single,
        }
    )
depth_table = pd.DataFrame(rows).round(2)
depth_table.to_csv(os.path.join(DATA_DIR, f"{opsim_name}_ddf_depth_summary_long.csv"), index=False)

# Wide tables: one row per DDF, one column per band
n_visits_table = depth_table.pivot(index="DDF", columns="band", values="N visits")
n_visits_table = n_visits_table.reindex(index=DDF_NAMES, columns=BANDS).fillna(0).astype(int)
n_visits_table.to_csv(os.path.join(DATA_DIR, f"{opsim_name}_ddf_n_visits_per_band.csv"))

final_depth_table = depth_table.pivot(index="DDF", columns="band", values="final coadded m5")
final_depth_table = final_depth_table.reindex(index=DDF_NAMES, columns=BANDS)
final_depth_table.to_csv(os.path.join(DATA_DIR, f"{opsim_name}_ddf_final_depth_per_band.csv"))

print("Number of visits per DDF and band")
display(n_visits_table)
print("Final coadded depth m5 (mag) per DDF and band")
display(final_depth_table)

## 4. Figure

### Figure 1 - Cumulative depth of each DDF versus date (`<run>_ddf_cumulative_depth.png`)

One panel per DDF and one curve per band. The horizontal axis is the calendar date; the dashed vertical lines mark the beginning of each survey year (Y1 ... Y10). A step in a curve is a season in which the DDF is observed in that band; a flat part is a period without visits in that band. The vertical axis is limited to 25-29 mag.

In [ ]:
def save_fig(fig, stem):
    # Each figure is saved as .png and .pdf in FIG_DIR
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG_DIR, f"{opsim_name}_{stem}.{ext}"), dpi=180, bbox_inches="tight")


def plot_ddfs(df_cum, ddflist):
    # One panel per DDF, one curve of cumulative depth per band
    fig, axes = plt.subplots(
        len(ddflist), 1, figsize=(12, 2 * len(ddflist)), sharex=True, constrained_layout=True
    )
    if len(ddflist) == 1:
        axes = [axes]

    for ax, ddf in zip(axes, ddflist):
        subset = df_cum[df_cum["ddf"] == ddf]

        for band in BANDS:
            v = subset[subset["filter"] == band]
            if v.empty:
                continue
            ax.plot(v["date"], v["m5_cum"], color=BAND_COLORS[band], label=band, linewidth=3)

        # Beginning of each survey year, and year label at the middle of the year
        for n in range(N_YEARS + 1):
            boundary = survey_year_date(n)
            ax.axvline(boundary, color="0.55", linestyle="--", linewidth=3, alpha=0.7)
            if n < N_YEARS:
                midpoint = boundary + (survey_year_date(n + 1) - boundary) / 2
                ax.text(
                    midpoint,
                    0.97,
                    f"Y{n + 1}",
                    transform=ax.get_xaxis_transform(),
                    ha="center",
                    va="top",
                    fontsize=9,
                    color="0.3",
                )

        ax.set_title(ddf, fontsize=15, loc="left")
        ax.set_ylabel(r"Cumulative $m_5$ (mag)", fontsize=12)
        ax.grid(True, alpha=0.25)
        ax.legend(title="LSST band", ncol=6, loc="lower right", frameon=True)
        ax.set_ylim(25.0, 29.0)

    axes[-1].set_xlabel("Date", fontsize=12)
    axes[-1].xaxis.set_major_locator(mdates.YearLocator())
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    axes[-1].set_xlim(
        datetime.combine(SURVEY_START, datetime.min.time()),
        datetime.combine(survey_year_date(N_YEARS), datetime.min.time()),
    )
    fig.suptitle(f"Cumulative DDF depth by band - {opsim_name}", fontsize=17)
    save_fig(fig, "ddf_cumulative_depth")
    return fig


fig1 = plot_ddfs(df_cum, DDF_NAMES)
plt.show()